In [1]:
%matplotlib tk
import scipy as sc
import numpy as np
import sympy as sp
import matplotlib as mpl
import matplotlib.pyplot as plt
import scienceplots

plt.style.use(['science','notebook', 'grid'])

In [2]:
N = 250

t, g = sp.symbols('t g')
m1, m2, b = sp.symbols('m1 m2 b')
L1, L2 = sp.symbols('L1 L2')
k1, k2 = sp.symbols('k1 k2')

In [3]:
the1, the2, r1, r2 = sp.symbols(r'\theta_1, \theta_2, r1, r2',cls=sp.Function)

the1 = the1(t)
the2 = the2(t)
r1 = r1(t)
r2 = r2(t)

In [4]:
the1_d = sp.diff(the1, t)
the2_d = sp.diff(the2, t)
the1_dd = sp.diff(the1_d, t)
the2_dd = sp.diff(the2_d, t)

r1_d = sp.diff(r1, t)
r2_d = sp.diff(r2, t)
r1_dd = sp.diff(r1_d, t)
r2_dd = sp.diff(r2_d, t)

In [5]:
x1 = r1 * sp.sin(the1)
y1 = -r1 * sp.cos(the1)
x2 = r1 * sp.sin(the1) + r2 * sp.sin(the2)
y2 = -r1 * sp.cos(the1) - r2 * sp.cos(the2)

In [6]:
# Kinetic Energy Term
T1 = 0.5 * m1 * (sp.diff(x1,t)**2+sp.diff(y1,t)**2)
T2 = 0.5 * m2 * (sp.diff(x2,t)**2+sp.diff(y2,t)**2)
T = T1 + T2

# Potential Energy Term
U1 = m1 * g * y1 + k1*(sp.sqrt(x1**2 + y1**2)-L1)**2/2 + k2 * (sp.sqrt((x1-x2)**2 + (y1-y2)**2)-L2)**2/2
U2 = m2 * g * y2 + k2 * (sp.sqrt((x1-x2)**2 + (y1-y2)**2)-L2)**2/2
U = U1 + U2

# Lagrangian
L = (T - U)#*sp.exp(b*t)

In [7]:
LE1 = sp.diff(L,the1) - sp.diff(sp.diff(L,the1_d),t).simplify()
LE2 = sp.diff(L,the2) - sp.diff(sp.diff(L,the2_d),t).simplify()
LE3 = sp.diff(L,r1) - sp.diff(sp.diff(L,r1_d),t).simplify()
LE4 = sp.diff(L,r2) - sp.diff(sp.diff(L,r2_d),t).simplify()


In [8]:
sols = sp.solve([LE1, LE2, LE3, LE4], (the1_dd, the2_dd, r1_dd, r2_dd), simplify = False, rational=False)

Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "C:\Users\hasan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sympy\core\assumptions.py", line 499, in getit
    return self._assumptions[fact]
           ~~~~~~~~~~~~~~~~~^^^^^^
KeyError: 'zero'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "C:\Users\hasan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\IPython\core\interactiveshell.py", line 3672, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "C:\Users\hasan\AppData\Local\Temp\ipykernel_562464\2243848239.py", line 1, in <module>
    sols = sp.solve([LE1, LE2, LE3, LE4], (the1_dd, the2_dd, r1_dd, r2_dd), simplify = False, rational=False)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  Fil

In [15]:
dw_1dt_f = sp.lambdify((t,g,m1,m2,L1,L2,k1,k2,b,the1,the2,the1_d,the2_d, r1, r2, r1_d, r2_d), sols[the1_dd])
dw_2dt_f = sp.lambdify((t,g,m1,m2,L1,L2,k1,k2,b,the1,the2,the1_d,the2_d, r1, r2, r1_d, r2_d), sols[the2_dd])
dthe1dt_f = sp.lambdify(the1_d,the1_d)
dthe2dt_f = sp.lambdify(the2_d,the2_d)

dv_1dt_f = sp.lambdify((t,g,m1,m2,L1,L2,k1,k2,b,the1,the2,the1_d,the2_d, r1, r2, r1_d, r2_d), sols[r1_dd])
dv_2dt_f = sp.lambdify((t,g,m1,m2,L1,L2,k1,k2,b,the1,the2,the1_d,the2_d, r1, r2, r1_d, r2_d), sols[r2_dd])
dr1dt_f = sp.lambdify(r1_d,r1_d)
dr2dt_f = sp.lambdify(r2_d,r2_d)

In [17]:
def dSdt(S, t , g, m1, m2, L1, L2, k1, k2, b):
    the1, w1, the2, w2, r1, v1, r2, v2 = S
    return [
        dthe1dt_f(w1),
        dw_1dt_f(t, g, m1, m2, L1, L2, k1, k2, b, the1, the2, w1, w2, r1, r2, v1, v2),
        dthe2dt_f(w2),
        dw_2dt_f(t, g, m1, m2, L1, L2, k1, k2, b, the1, the2, w1, w2, r1, r2, v1, v2),
        dr1dt_f(v1),
        dv_1dt_f(t, g, m1, m2, L1, L2, k1, k2, b, the1, the2, w1, w2, r1, r2, v1, v2),
        dr2dt_f(v2),
        dv_2dt_f(t, g, m1, m2, L1, L2, k1, k2, b, the1, the2, w1, w2, r1, r2, v1, v2)
    ]

In [87]:
t = np.linspace(0, 10, N)
g = 0#9.81
m1 = 2
m2 = 1
L1 = 1
L2 = 1
k1 = 35
k2 = 35
b = 0
ans = sc.integrate.odeint(dSdt, y0=[-0, 0.3, 0.1, 0.2, 0.9, 0, 1.8, 0], t=t, args=(g,m1,m2,L1,L2,k1,k2,b))

In [88]:
the1 = ans.T[0]
the2 = ans.T[2]
r1 = ans.T[4]
r2 = ans.T[6]
plt.plot(t,r2)

In [89]:
def get_x1y1x2y2(t, the1,the2,r1,r2):
    return (
        r1 * np.sin(the1),
        -r1 * np.cos(the1),
        r1 * np.sin(the1) + r2 * np.sin(the2),
        -r1 * np.cos(the1) - r2 * np.cos(the2)
    )

x1, y1, x2, y2 = get_x1y1x2y2(t, ans.T[0], ans.T[2], ans.T[4], ans.T[6])

In [90]:
def animate(i):
    ln1.set_data([0, x1[i], x2[i]], [0, y1[i], y2[i]])

In [ ]:
from matplotlib import pyplot as plt, animation

fig, ax = plt.subplots()
ax.set_facecolor('k')
ax.set(xlim=(-4, 4), ylim=(-4, 4))
ln1, = plt.plot([], [], 'ro--', markersize=8)

ani = animation.FuncAnimation(fig, animate, frames = N-1, interval = 50)
#ani.save(filename="/Users/hasan/Python Animations/Double Coupled Spring Pendulum 0G.gif", writer="pillow")
#plt.show(ani)